# Detecção de Gráficos em PDFs — Comparação de 4 Métodos (10 PDFs difíceis)

**Foco:** os 10 PDFs onde os métodos anteriores mais falharam.

**Métodos comparados:**

| # | Método | Tipo | VRAM | Saída |
|---|---|---|---|---|
| 1 | **Chandra OCR 2** | VLM (4B) | ~8 GB | HTML com `<img>`, `<table>`, `<math>` |
| 2 | **PyMuPDF `get_images`** | Heurístico | 0 | imagens raster embutidas |
| 3 | **PyMuPDF `get_drawings`** | Heurístico | 0 | clusters DBSCAN de drawings vetoriais |
| 4 | **GLM-OCR (zai-org)** | VLM (0.9B) | ~3 GB | Markdown — combinando 3 prompts (Document/Table/Text) |

**Observação sobre o GLM-OCR:** o prompt único `Text Recognition:` (versão anterior) descartava completamente figuras e tabelas estruturadas. Esta versão executa **3 prompts em sequência** por página (`Document Parsing:`, `Table Recognition:`, `Text Recognition:`) e combina os outputs, tornando o GLM comparável aos demais métodos no quesito multimodal. Isso aumenta o tempo de inferência ~3x mas torna a comparação metodologicamente justa.

**dots.ocr foi removido** desta análise devido a incompatibilidades técnicas persistentes (`cache_position` bug com `transformers >= 4.45`) que não puderam ser resolvidas em ambiente Colab.

**Estratégia de VRAM:** os modelos VLM são carregados **um por vez** e descarregados antes do próximo, pra caber numa T4 (16 GB).

> Ative GPU em *Runtime → Change runtime type → GPU*.


## 1. Instalação

In [ ]:
# Dependências comuns
!pip install -q pymupdf scikit-learn pandas matplotlib pillow beautifulsoup4 ipywidgets

# Transformers e utils
!pip install -q -U transformers accelerate


In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}", end='')
if torch.cuda.is_available():
    print(f"  |  {torch.cuda.get_device_name(0)}  |  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")


## 2. Configuração de pastas + lista dos 10 PDFs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

INPUT_DIR     = Path('/content/drive/MyDrive/Chandra2/PDF')
OUTPUT_ROOT   = Path('/content/drive/MyDrive/Chandra2/output')
CHANDRA_DIR   = OUTPUT_ROOT / 'chandra'
PYMUPDF_RAST  = OUTPUT_ROOT / 'pymupdf_raster'
PYMUPDF_VECT  = OUTPUT_ROOT / 'pymupdf_vector'
GLM_DIR       = OUTPUT_ROOT / 'glm_ocr'
GT_CSV        = OUTPUT_ROOT / 'avaliacao_manual.csv'

GLM_DIR.mkdir(parents=True, exist_ok=True)

# ===== Os 10 PDFs alvo =====
TARGET_NAMES = [
    'W3132421076.pdf',
    'Mattos(2018)-IA 163 - Artigo Citros-Cafe - Fernanda Bochi Dos Santos.pdf',
    'W3110745114.pdf',
    'W4309496579.pdf',
    'Yamane(2022)-horticulturae-08-01126 - Janaina Lais Pacheco Lara Morandin (1).pdf',
    'W3138442591.pdf',
    'grafico de radar.pdf',
    'W4323846702.pdf',
    'Mattos(2018)-TodaFruta2 - Fernanda Bochi Dos Santos.pdf',
    'W3216639369.pdf',
]

pdfs = [INPUT_DIR / n for n in TARGET_NAMES if (INPUT_DIR / n).exists()]
missing = [n for n in TARGET_NAMES if not (INPUT_DIR / n).exists()]

print(f"PDFs encontrados: {len(pdfs)} / {len(TARGET_NAMES)}")
for p in pdfs:
    print(f"  ✓ {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
for n in missing:
    print(f"  ✗ {n}  (NÃO ENCONTRADO)")


## 3. Métodos 1–3 — reusar resultados do Drive

Estes três já rodaram antes. Apenas leia os outputs salvos.

In [ ]:
import fitz, io
from PIL import Image
from bs4 import BeautifulSoup
from collections import Counter
import json

# ----- Chandra (lê HTML + arquivos extraídos da subpasta) -----
def chandra_counts(pdf_stem):
    sub = CHANDRA_DIR / pdf_stem
    html_p = sub / f"{pdf_stem}.html"
    cnt = Counter()
    if html_p.exists():
        soup = BeautifulSoup(html_p.read_text(encoding='utf-8'), 'html.parser')
        cnt['image']    = len(soup.find_all('img'))
        cnt['table']    = len(soup.find_all('table'))
        cnt['equation'] = len(soup.find_all('math'))
    cnt['_imgs_extracted'] = (
        len(list(sub.glob('*.webp'))) +
        len(list(sub.glob('*.png')))  +
        len(list(sub.glob('*.jpg')))
    ) if sub.exists() else 0
    return cnt

chandra_results = {pdf.name: chandra_counts(pdf.stem) for pdf in pdfs}

# ----- PyMuPDF raster -----
MIN_RASTER_SIDE = 150
def extract_raster(pdf_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob('*'): f.unlink()
    doc = fitz.open(str(pdf_path))
    found, seen = [], set()
    for page_num, page in enumerate(doc, start=1):
        for info in page.get_images(full=True):
            xref = info[0]
            if xref in seen: continue
            seen.add(xref)
            try:
                base = doc.extract_image(xref)
                img = Image.open(io.BytesIO(base['image']))
                if min(img.size) < MIN_RASTER_SIDE: continue
                fname = out_dir / f"p{page_num:03d}_xref{xref}.png"
                img.save(fname)
                found.append({'page': page_num, 'path': str(fname),
                              'w': img.width, 'h': img.height})
            except Exception: continue
    doc.close()
    return found

raster_results = {pdf.name: extract_raster(pdf, PYMUPDF_RAST/pdf.stem) for pdf in pdfs}

# ----- PyMuPDF vetorial -----
import numpy as np
from sklearn.cluster import DBSCAN

def detect_vector(pdf_path, out_dir,
                  eps_pt=25, min_samples=15,
                  min_area_frac=0.03, max_area_frac=0.80,
                  aspect_min=0.20, aspect_max=5.00, render_dpi=150):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob('*'): f.unlink()
    doc = fitz.open(str(pdf_path))
    found = []
    for page_num, page in enumerate(doc, start=1):
        drawings = page.get_drawings()
        if len(drawings) < min_samples: continue
        bboxes, centers = [], []
        for d in drawings:
            r = d.get('rect')
            if r is None: continue
            bboxes.append([r.x0, r.y0, r.x1, r.y1])
            centers.append([(r.x0+r.x1)/2, (r.y0+r.y1)/2])
        if len(centers) < min_samples: continue
        bboxes = np.array(bboxes); centers = np.array(centers)
        labels = DBSCAN(eps=eps_pt, min_samples=min_samples).fit_predict(centers)
        page_area = page.rect.width * page.rect.height
        ci = 0
        for lab in sorted(set(labels)):
            if lab == -1: continue
            cb = bboxes[labels == lab]
            x0,y0,x1,y1 = cb[:,0].min(), cb[:,1].min(), cb[:,2].max(), cb[:,3].max()
            w,h = x1-x0, y1-y0
            if w<=0 or h<=0: continue
            af = (w*h)/page_area
            if not (min_area_frac<=af<=max_area_frac): continue
            ar = w/h
            if not (aspect_min<=ar<=aspect_max): continue
            mat = fitz.Matrix(render_dpi/72, render_dpi/72)
            pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(x0,y0,x1,y1))
            ci += 1
            fname = out_dir / f"p{page_num:03d}_c{ci:02d}.png"
            Image.open(io.BytesIO(pix.tobytes('png'))).save(fname)
            found.append({'page': page_num, 'path': str(fname)})
    doc.close()
    return found

vector_results = {pdf.name: detect_vector(pdf, PYMUPDF_VECT/pdf.stem) for pdf in pdfs}

print("Resultados Chandra/PyMuPDF carregados:")
for pdf in pdfs:
    c = chandra_results[pdf.name]
    print(f"  {pdf.name[:60]:<60}  Chandra: img={c['image']}, tab={c['table']}  | "
          f"raster={len(raster_results[pdf.name])}  | vetor={len(vector_results[pdf.name])}")


## 4. Helper — renderizar páginas dos PDFs em PNG

O GLM-OCR trabalha em imagens, não PDFs.

In [ ]:
def render_pdf_pages(pdf_path, dpi=150):
    doc = fitz.open(str(pdf_path))
    pages = []
    for p in doc:
        pix = p.get_pixmap(dpi=dpi)
        pages.append(Image.open(io.BytesIO(pix.tobytes('png'))))
    doc.close()
    return pages

print("Renderizando páginas dos 10 PDFs...")
all_pages = {}
for pdf in pdfs:
    all_pages[pdf.name] = render_pdf_pages(pdf, dpi=150)
    print(f"  {pdf.name}: {len(all_pages[pdf.name])} páginas")


In [ ]:
# Helper pra liberar VRAM entre modelos
import gc

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    if torch.cuda.is_available():
        free_gb = torch.cuda.mem_get_info()[0]/1e9
        print(f"  VRAM livre após cleanup: {free_gb:.1f} GB")


## 5. Método 4 — GLM-OCR (versão com 3 prompts)

**Mudança importante**: roda **3 prompts por página** em vez de 1:

- `Document Parsing:` — captura estrutura geral, incluindo figuras como `![]()`
- `Table Recognition:` — extrai tabelas como HTML estruturado (`<table>`)
- `Text Recognition:` — texto corrido com fórmulas em LaTeX

Os 3 outputs são combinados em um markdown único delimitado por comentários HTML, e o regex de parsing pega tudo. Tempo de inferência ~3x maior, mas torna o GLM comparável aos demais métodos no quesito multimodal.

In [ ]:
# Backup do GLM antigo (versão Text Recognition: only) antes de sobrescrever
import shutil
GLM_DIR_BACKUP = OUTPUT_ROOT / 'glm_ocr_text_only_BACKUP'
if GLM_DIR.exists() and not GLM_DIR_BACKUP.exists():
    shutil.copytree(GLM_DIR, GLM_DIR_BACKUP)
    print(f"✓ Backup do GLM antigo salvo: {GLM_DIR_BACKUP}")
else:
    if GLM_DIR_BACKUP.exists():
        print(f"  (backup já existe em {GLM_DIR_BACKUP})")
    else:
        print(f"  (não há GLM antigo para fazer backup)")


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import time, re

print("Carregando GLM-OCR (zai-org/GLM-OCR)...")
GLM_MODEL_ID = "zai-org/GLM-OCR"
glm_processor = AutoProcessor.from_pretrained(GLM_MODEL_ID, trust_remote_code=True)
glm_model = AutoModelForImageTextToText.from_pretrained(
    GLM_MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
glm_model.eval()
print("✓ GLM-OCR carregado")


In [ ]:
# === Versão melhorada — combina 3 prompts pra extração multimodal completa ===

GLM_PROMPTS = {
    'document': "Document Parsing:",   # estrutura geral + figuras como ![]()
    'table':    "Table Recognition:",  # tabelas como HTML estruturado
    'text':     "Text Recognition:",   # texto corrido com fórmulas LaTeX
}

def glm_parse_page(pil_image):
    """Roda 3 prompts no GLM-OCR e combina os outputs em um único markdown."""
    tmp_path = "/tmp/glm_page.png"
    pil_image.save(tmp_path)

    outputs = {}
    for label, prompt_text in GLM_PROMPTS.items():
        try:
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "url": tmp_path},
                    {"type": "text", "text": prompt_text},
                ],
            }]
            inputs = glm_processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt"
            ).to(glm_model.device)
            inputs.pop("token_type_ids", None)

            with torch.no_grad():
                gen = glm_model.generate(**inputs, max_new_tokens=4096, do_sample=False)
            out = glm_processor.decode(
                gen[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            )
            outputs[label] = out
        except Exception as e:
            outputs[label] = f"<!-- erro no prompt {label}: {e} -->"

    # Combina os 3 outputs num markdown estruturado, delimitado por comentários
    combined = []
    combined.append("<!-- BEGIN document parsing -->")
    combined.append(outputs['document'])
    combined.append("<!-- END document parsing -->\n")
    combined.append("<!-- BEGIN table recognition -->")
    combined.append(outputs['table'])
    combined.append("<!-- END table recognition -->\n")
    combined.append("<!-- BEGIN text recognition -->")
    combined.append(outputs['text'])
    combined.append("<!-- END text recognition -->\n")

    return "\n".join(combined)

def count_elements_in_markdown(md_text):
    n_images   = len(re.findall(r'!\[[^\]]*\]\([^)]+\)', md_text))
    n_tables   = len(re.findall(r'<table[^>]*>', md_text, flags=re.IGNORECASE))
    md_table_blocks = re.findall(r'(?:^\|[^\n]+\|\n){2,}', md_text, flags=re.MULTILINE)
    n_tables += len(md_table_blocks)
    n_equations = len(re.findall(r'\$\$[^$]+\$\$', md_text))
    return {'image': n_images, 'table': n_tables, 'equation': n_equations}


In [ ]:
# === TESTE em 1 página antes do loop completo ===
test_pdf = pdfs[0]
test_page = all_pages[test_pdf.name][0]

print(f"=== Teste GLM-OCR com 3 prompts ===")
print(f"PDF: {test_pdf.name}, página 1\n")

t0 = time.time()
result = glm_parse_page(test_page)
print(f"Tempo: {time.time()-t0:.1f}s")
print(f"Tamanho total: {len(result)} chars\n")

# Confere o que veio em cada seção
print(f"Marcações encontradas no output combinado:")
print(f"  ![](...) figuras:       {result.count('![')}")
print(f"  <table> tabelas HTML:   {result.count('<table')}")
print(f"  |---|... tabelas md:    {result.count('|---')}")
print(f"  $$...$$ equações:       {result.count('$$')//2}")

# Mostra primeiros 1500 chars
print(f"\n=== Output (primeiros 1200 chars) ===")
print(result[:1200])


In [ ]:
# === Loop completo nos 10 PDFs (~1.5-2h em T4) ===
glm_results = {}
t0 = time.time()

for pdf in pdfs:
    out_dir = GLM_DIR / pdf.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    pages = all_pages[pdf.name]
    full_md, totals = [], Counter()
    print(f"\n→ {pdf.name} ({len(pages)} páginas)")
    for i, page_img in enumerate(pages, 1):
        try:
            md_text = glm_parse_page(page_img)
            full_md.append(f"--- PAGE {i} ---\n{md_text}\n")
            cnt = count_elements_in_markdown(md_text)
            for k, v in cnt.items(): totals[k] += v
            print(f"   p{i}: img={cnt['image']}, tab={cnt['table']}, eq={cnt['equation']}")
        except Exception as e:
            print(f"   p{i}: ERRO {e}")
    (out_dir / "output.md").write_text("\n".join(full_md), encoding='utf-8')
    glm_results[pdf.name] = dict(totals)
    print(f"   TOTAL: {dict(totals)}")
print(f"\n=== GLM-OCR concluído em {(time.time()-t0)/60:.1f} min ===")


In [ ]:
# Libera VRAM
del glm_model, glm_processor
free_vram()


## 6. Tabela bruta — quantos elementos cada método encontrou

In [ ]:
import pandas as pd

def n_chandra_visuals(name): return chandra_results[name].get('image', 0)
def n_chandra_tables(name):  return chandra_results[name].get('table', 0)

def n_glm_visuals(name): return glm_results.get(name, {}).get('image', 0)
def n_glm_tables(name):  return glm_results.get(name, {}).get('table', 0)

raw_rows = []
for pdf in pdfs:
    n = pdf.name
    raw_rows.append({
        'pdf': n[:35],
        'Chandra <img>': n_chandra_visuals(n),
        'Chandra <tbl>': n_chandra_tables(n),
        'PyMuPDF rast':  len(raster_results[n]),
        'PyMuPDF vec':   len(vector_results[n]),
        'GLM ![]':       n_glm_visuals(n),
        'GLM <tbl>':     n_glm_tables(n),
    })
raw_df = pd.DataFrame(raw_rows)
print(raw_df.to_string(index=False))

raw_df.to_csv(OUTPUT_ROOT / 'metodos_4_brutos_top10.csv', index=False)
print(f"\nSalvo em: {OUTPUT_ROOT/'metodos_4_brutos_top10.csv'}")


## 7. Ranking — qual método foi melhor nos 10 PDFs difíceis?

In [ ]:
import numpy as np

gt_df = pd.read_csv(GT_CSV).set_index('pdf')
common = [p.name for p in pdfs if p.name in gt_df.index]
print(f"Avaliações manuais carregadas: {len(common)}/{len(pdfs)} PDFs\n")

gt_total_vec    = [int(gt_df.loc[n, 'gt_total'])     for n in common]
gt_graf_img_vec = [int(gt_df.loc[n, 'gt_graficos']) + int(gt_df.loc[n, 'gt_imagens']) for n in common]
gt_tab_vec      = [int(gt_df.loc[n, 'gt_tabelas'])   for n in common]

chand_vis = [n_chandra_visuals(n) for n in common]
chand_tab = [n_chandra_tables(n)  for n in common]
chand_tot = [a+b for a,b in zip(chand_vis, chand_tab)]

rast_tot  = [len(raster_results[n]) for n in common]
vec_tot   = [len(vector_results[n]) for n in common]

glm_vis = [n_glm_visuals(n) for n in common]
glm_tab = [n_glm_tables(n)  for n in common]
glm_tot = [a+b for a,b in zip(glm_vis, glm_tab)]

def metrics(detected, gt):
    detected, gt = np.array(detected), np.array(gt)
    err = detected - gt
    return {
        'detectado': int(detected.sum()),
        'GT':        int(gt.sum()),
        'MAE':       round(float(np.abs(err).mean()), 2),
        'viés':      round(float(err.mean()), 2),
        'perfeitos': int((err == 0).sum()),
    }

results = {
    'Chandra (total)':       metrics(chand_tot, gt_total_vec),
    'PyMuPDF raster':        metrics(rast_tot,  gt_total_vec),
    'PyMuPDF vetorial':      metrics(vec_tot,   gt_total_vec),
    'GLM-OCR (total)':       metrics(glm_tot,   gt_total_vec),
    'Chandra (img/fig)':     metrics(chand_vis, gt_graf_img_vec),
    'GLM-OCR (img/fig)':     metrics(glm_vis,   gt_graf_img_vec),
    'Chandra (table)':       metrics(chand_tab, gt_tab_vec),
    'GLM-OCR (table)':       metrics(glm_tab,   gt_tab_vec),
}

ranking_df = pd.DataFrame(results).T.sort_values('MAE')
print("=== RANKING DOS 10 PDFs DIFÍCEIS (menor MAE = melhor) ===\n")
print(ranking_df.to_string())

ranking_df.to_csv(OUTPUT_ROOT / 'ranking_4_metodos_top10.csv')
print(f"\nSalvo em: {OUTPUT_ROOT/'ranking_4_metodos_top10.csv'}")


In [ ]:
# Gráfico comparativo final
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(common))
w = 0.16

ax.bar(x - 2*w, gt_total_vec, w, label='GT', color='black')
ax.bar(x - 1*w, chand_tot,    w, label='Chandra')
ax.bar(x      , rast_tot,     w, label='PyMuPDF raster')
ax.bar(x + 1*w, vec_tot,      w, label='PyMuPDF vetor')
ax.bar(x + 2*w, glm_tot,      w, label='GLM-OCR')

ax.set_xticks(x)
ax.set_xticklabels([n.replace('.pdf','')[:18] for n in common],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('# elementos detectados')
ax.set_title('Comparação dos 4 métodos nos 10 PDFs difíceis (GLM com 3 prompts)')
ax.legend(ncol=5, fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'comparacao_4_metodos_top10.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nFigura salva: {OUTPUT_ROOT/'comparacao_4_metodos_top10.png'}")


## 8. Conclusão e próximos passos

Compare os números com a versão anterior do GLM-OCR (que usava só `Text Recognition:`):

- **GLM antes (1 prompt):** detectava 1 figura e 1 tabela nos 10 PDFs.
- **GLM agora (3 prompts):** captura figuras (`![]()`), tabelas HTML (`<table>`) e mantém o texto de qualidade.

A diferença mostra que a configuração do prompt é decisiva para extração multimodal — não basta usar um modelo VLM com capacidade, é preciso ativá-la corretamente.

**Próximo:** rode o Notebook 5 (`comparacao_qualidade_4dim.ipynb`) com os novos outputs do GLM no Drive. Os heatmaps Jaccard agora terão GLM como participante ativo nas 4 dimensões.